In [35]:
import os
import numpy as np

#Open libsgma, onyxia
import sys
sys.path.append('/home/onyxia/work')
from libsigma import classification as cla
from libsigma import read_and_write as rw
from libsigma import plots


from osgeo import gdal
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.tree import DecisionTreeClassifier # ou RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import geopandas as gpd #ler arquivos vetoriais



#from IPython.display import display

In [38]:
# 1. Defina a lista de caminhos de arquivo
# **Substitua este caminho base pelo caminho real onde seus arquivos estão localizados**
base_dir = '/home/onyxia/work/data/projet_eval/'

# Lista dos arquivos que você deseja abrir (exemplo)
filenames = [
    'pyrenees_23-24_B02.tif',
    'pyrenees_23-24_B03.tif',
    'pyrenees_23-24_B04.tif',
    'pyrenees_23-24_B05.tif',
    'pyrenees_23-24_B06.tif',
    'pyrenees_23-24_B07.tif',
    'pyrenees_23-24_B08.tif',
    'pyrenees_23-24_B8A.tif',
    'pyrenees_23-24_B11.tif',
    'pyrenees_23-24_B12.tif'
    ]

# Crie um dicionário para armazenar os caminhos e os objetos GDAL abertos
# 'caminho_do_arquivo': objeto_gdal
gdal_datasets = {}

# 2. Itere sobre a lista usando um loop 'for'
for filename in filenames:
    # 3. Construa o caminho completo do arquivo
    full_path = os.path.join(base_dir, filename)

    # 4. Abra o arquivo
    try:
        # Usando gdal.GA_ReadOnly se você só for ler, ou GA_Update se for modificar
        data_set = gdal.Open(full_path, gdal.GA_Update)

        if data_set is not None:
            print(f"✅ Arquivo aberto com sucesso: {full_path}")
            # 5. Salve o caminho completo e o objeto data_set no dicionário
            gdal_datasets[full_path] = data_set
            print(f"Número de bandas: {primeiro_dataset.RasterCount}")
            
        else:
            print(f"❌ Não foi possível abrir o arquivo: {full_path}")

    except Exception as e:
        print(f"🛑 Erro ao tentar abrir {full_path}: {e}")

# Exemplo de como acessar um dos datasets abertos:
# (Isto é útil se você precisar processar os dados mais tarde)
if gdal_datasets:
    # Obtém o caminho da primeira chave (primeiro arquivo na lista)
    primeiro_caminho = list(gdal_datasets.keys())[0]
    primeiro_dataset = gdal_datasets[primeiro_caminho]

    print("\n--- Informações do primeiro arquivo aberto ---")
    print(f"Caminho: {primeiro_caminho}")
    print(f"Número de bandas: {primeiro_dataset.RasterCount}")
    print(f"Driver: {primeiro_dataset.GetDriver().LongName}")

# **IMPORTANTE: Feche os datasets quando terminar de usá-los!**
# Isso é crucial para liberar recursos e garantir que as alterações sejam salvas (se GA_Update foi usado).
for path, dataset in gdal_datasets.items():
    dataset = None # Fechar a referência ao dataset
    print(f"Fechado: {path}")



✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B02.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B03.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B04.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B05.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B06.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B07.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B08.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B8A.tif
Número de bandas: 15
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B11.tif
Número de bandas: 15
✅ Arquivo aberto co

In [39]:
def listar_colunas_do_shapefile(base_data, samples_file_name):
    """
    Carrega o arquivo shapefile e imprime todos os nomes das colunas
    (incluindo a coluna 'geometry').
    """
    print("\n--- INICIANDO VERIFICAÇÃO DE COLUNAS VETORIAIS ---")

    # 1. Constrói o caminho completo do arquivo
    samples_file = os.path.join(base_data, samples_file_name)
    print(f"Tentando carregar o arquivo: {samples_file}")

    # 2. Carrega o arquivo usando GeoPandas
    try:
        # Tenta carregar o arquivo
        gdf = gpd.read_file(samples_file)
        print(f"✅ GeoDataFrame carregado com sucesso ({len(gdf)} feições).")

    except Exception as e:
        print(f"❌ Erro ao carregar o arquivo vetorial: {e}")
        # Retorna uma lista vazia em caso de falha
        return []

    # 3. Extrai e exibe os nomes das colunas
    colunas = gdf.columns.tolist()
    
    print("\n==============================================")
    print("📢 NOMES DAS COLUNAS DISPONÍVEIS NO SHAPEFILE:")
    print("==============================================")
    for coluna in colunas:
        print(f" - {coluna}")
    print("==============================================")
    
    return colunas

# --- BLOCO PRINCIPAL DE EXECUÇÃO ---

if __name__ == "__main__":
    
    # --- PARTE VETORIAL: Configurações ---
    
    # 🚨 Use o caminho exato onde o seu arquivo está.
    base_data_vector = "/home/onyxia/work/data/projet_eval"
    samples_file_name_vector = "PI_strates_pyrenees_32630.shp"
    
    # Executa a função
    colunas_encontradas = listar_colunas_do_shapefile(
        base_data_vector,
        samples_file_name_vector
    )
    
    # Exemplo de como usar a informação:
    if colunas_encontradas:
        print(f"\nTotal de colunas encontradas: {len(colunas_encontradas)}")
        
        # Se você precisar encontrar o nome da sua coluna de classe, por exemplo:
        # nome_da_classe = colunas_encontradas[1] 
        # print(f"Segunda coluna (exemplo de classe): {nome_da_classe}")


--- INICIANDO VERIFICAÇÃO DE COLUNAS VETORIAIS ---
Tentando carregar o arquivo: /home/onyxia/work/data/projet_eval/PI_strates_pyrenees_32630.shp
✅ GeoDataFrame carregado com sucesso (206 feições).

📢 NOMES DAS COLUNAS DISPONÍVEIS NO SHAPEFILE:
 - id
 - strate
 - comment
 - geometry

Total de colunas encontradas: 4


In [40]:
# --- VARIÁVEL DE CONFIGURAÇÃO GLOBAL (Define a coluna usada para a classificação) ---
COLUNA_CLASSE = 'strate'

# --- DEFINIÇÕES DE FUNÇÕES CORRIGIDAS ---

def count_polygons_by_class(gdf, col_name=COLUNA_CLASSE):
    """Conta o número de feições (polígonos) por classe no GeoDataFrame."""
    # CORREÇÃO: Usa o nome da coluna definido em COLUNA_CLASSE (ou passado como argumento)
    return gdf[col_name].value_counts()

def rasterize_samples(gdf, col_name=COLUNA_CLASSE):
    """Simula a rasterização e contagem de pixels."""
    # CORREÇÃO: Usa o nome da coluna definido em COLUNA_CLASSE
    counts = gdf[col_name].value_counts()
    return counts * 100 # Simulação para pixels


def processar_e_visualizar_dados_vetoriais(base_data, results_dir, samples_file_name, col_class_name):
    """
    Função principal para carregar dados vetoriais, processar contagens e gerar visualizações.
    """
    print("\n--- INICIANDO PROCESSAMENTO VETORIAL ---")

    # 1. Configurar diretórios de saída
    fig_dir = os.path.join(results_dir, "figure")
    os.makedirs(results_dir, exist_ok=True)
    os.makedirs(fig_dir, exist_ok=True)
    print(f"Diretórios criados/verificados: {results_dir} e {fig_dir}")

    # 2. Carregar o arquivo GeoJSON/Shapefile
    samples_file = os.path.join(base_data, samples_file_name)
    try:
        gdf = gpd.read_file(samples_file)
        print(f"✅ GeoDataFrame carregado com {len(gdf)} feições.")
        print(f"Colunas disponíveis: {gdf.columns.tolist()}")
    except Exception as e:
        print(f"❌ Erro ao carregar o arquivo vetorial: {e}")
        return

    # 3. Processamento e Visualização de Contagem de Polígonos
    print(f"\nContando feições pela coluna: '{col_class_name}'")
    poly_counts = count_polygons_by_class(gdf, col_class_name)
    plot_bar_chart(poly_counts, fig_dir, "poly", "polygones", col_class_name)
    display(poly_counts)

    # 4. Processamento e Visualização de Contagem de Pixels (Rasterização)
    pix_counts = rasterize_samples(gdf, col_class_name)
    plot_bar_chart(pix_counts, fig_dir, "pix", "pixels", col_class_name)
    display(pix_counts)

    print("--- PROCESSAMENTO VETORIAL CONCLUÍDO ---")


def plot_bar_chart(counts_series, fig_dir, prefix, label_y, col_class_name):
    """
    Função auxiliar para gerar e salvar um gráfico de barras.
    Inclui o nome da coluna no título.
    """
    
    file_name = f"diag_baton_nb_{prefix}_by_{col_class_name}.png"
    
    plt.figure(figsize=(8,5))
    plt.bar(counts_series.index, counts_series.values)
    plt.xlabel(col_class_name.capitalize()) # Exibe 'Strate' no eixo X
    plt.ylabel(f"Nombre de {label_y}")
    plt.title(f"Nombre de {label_y} par {col_class_name}") # Título corrigido
    
    save_path = os.path.join(fig_dir, file_name)
    plt.savefig(save_path)
    print(f"Gráfico salvo em: {save_path}")
    
    # plt.show() # Descomente se quiser exibir o gráfico imediatamente
    plt.close()


# --- FUNÇÕES GDAL (Mantidas, pois não foram a causa do erro) ---

def abrir_datasets_gdal(base_dir, filenames):
    # ... (Código idêntico ao anterior)
    print("\n--- INICIANDO ABERTURA DE DATASETS GDAL ---")
    gdal_datasets = {}

    for filename in filenames:
        full_path = os.path.join(base_dir, filename)

        try:
            data_set = gdal.Open(full_path, gdal.GA_Update)

            if data_set is not None:
                print(f"✅ Arquivo aberto com sucesso: {full_path}")
                gdal_datasets[full_path] = data_set
            else:
                print(f"❌ Não foi possível abrir o arquivo (dataset vazio): {full_path}")

        except Exception as e:
            print(f"🛑 Erro ao tentar abrir {full_path}: {e}")

    print("--- ABERTURA DE DATASETS GDAL CONCLUÍDA ---")
    return gdal_datasets


def fechar_datasets_gdal(gdal_datasets):
    # ... (Código idêntico ao anterior)
    print("\n--- FECHANDO DATASETS GDAL ---")
    for path, dataset in gdal_datasets.items():
        dataset = None
        print(f"Fechado: {path}")
    print("--- DATASETS GDAL FECHADOS ---")


# --- BLOCO PRINCIPAL DE EXECUÇÃO CORRIGIDO ---

if __name__ == "__main__":
    
    # --- PARTE 1: Configurações e Execução GDAL (Raster) ---
    base_dir_raster = '/home/onyxia/work/data/projet_eval'
    filenames_raster = [
        'pyrenees_23-24_B02.tif', 'pyrenees_23-24_B03.tif', 'pyrenees_23-24_B04.tif',
        'pyrenees_23-24_B05.tif', 'pyrenees_23-24_B06.tif', 'pyrenees_23-24_B07.tif',
        'pyrenees_23-24_B08.tif', 'pyrenees_23-24_B8A.tif', 'pyrenees_23-24_B11.tif',
        'pyrenees_23-24_B12.tif' 
    ]

    datasets_abertos = abrir_datasets_gdal(base_dir_raster, filenames_raster)
    fechar_datasets_gdal(datasets_abertos)


    # --- PARTE 2: Configurações e Execução GeoPandas/Matplotlib (Vetorial) ---
    
    base_data_vector = "/home/onyxia/work/data/projet_eval"
    results_dir_vector = "results"
    samples_file_name_vector = "PI_strates_pyrenees_32630.shp"
    
    processar_e_visualizar_dados_vetoriais(
        base_data_vector,
        results_dir_vector,
        samples_file_name_vector,
        col_class_name=COLUNA_CLASSE # Passando o nome da coluna para a função
    )


--- INICIANDO ABERTURA DE DATASETS GDAL ---
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B02.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B03.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B04.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B05.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B06.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B07.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B08.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B8A.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B11.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B12.tif
--- ABERTURA DE DATASETS GDAL CONCLUÍDA ---

--- FECHANDO DATASETS GDAL ---

strate
3    78
2    69
4    31
1    28
Name: count, dtype: int64

Gráfico salvo em: results/figure/diag_baton_nb_pix_by_strate.png


strate
3    7800
2    6900
4    3100
1    2800
Name: count, dtype: int64

--- PROCESSAMENTO VETORIAL CONCLUÍDO ---
